# zagg TL;DR — AOI in, photon tensors out

The whole pipeline in one pass, timed per phase: **query** (NASA CMR → shard map), **build** (one Lambda per shard writes the store), **read** (masked shards → 128×128×64 signal-photon tensors → `.npz` on disk). Defaults are provided for everything except the AOI. The full story, plots included, lives in `01_query` / `02_write` / `03_read`.

In [1]:
# %pip install "zagg[catalog]" "moczarr>=0.4.0"
from datetime import date
from pathlib import Path

In [2]:
import os, time
import json

import botocore.session as _bs
import numpy as np

# reader
from moczarr.convention import morton_decimal
from moczarr.hhdc import read_tensors
from moczarr.open import open_leaf
from moczarr.store import read_manifest

# writer + query
from zagg.catalog import load_polygon
from zagg.catalog.shardmap import ShardMap
from zagg.catalog.sources import CMRSource, Query
from zagg.client import Run
from zagg.config import default_config
from zagg.grids import from_config

In [3]:
# sliderule auth
if "nasa" in (_bs.Session().full_config.get("profiles") or {}):
    os.environ.setdefault("AWS_PROFILE", "nasa")

# NASA Earthdata auth: the build phase calls earthaccess.login(), which expects
# credentials in the standard locations -- EARTHDATA_USERNAME/EARTHDATA_PASSWORD
# env vars, or ~/.netrc ("machine urs.earthdata.nasa.gov login <user> password <pw>").
# Without either it falls back to an interactive prompt.

# timer
timings = {}

class stage:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.perf_counter()
    def __exit__(self, *exc):
        timings[self.name] = round(time.perf_counter() - self.t0, 2)
        print(f"[{self.name}] {timings[self.name]:.1f}s")

In [4]:
# the AOI: raw GeoJSON (a bounding box is just a closed 5-point ring) --
# or a path to a .geojson file, or a packaged demo AOI: demo_aoi("serc")
AOI = {
    "type": "Polygon",
    "coordinates": [[
        [-76.62, 38.85],   # west, south
        [-76.50, 38.85],   # east, south
        [-76.50, 38.92],   # east, north
        [-76.62, 38.92],   # west, north
        [-76.62, 38.85],   # close the ring
    ]],
}

DATES = ("2018-10-13", date.today().isoformat())
STORE = "s3://sliderule-public/zagg-demo/tldr.zarr"  # use a FRESH prefix per test run
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

In [5]:
# raw GeoJSON is materialized to a file; paths pass through -- one code path after this
AOI_PATH = AOI
if isinstance(AOI, dict):
    AOI_PATH = str(OUT / "aoi.geojson")
    Path(AOI_PATH).write_text(json.dumps(AOI))
PARTS = load_polygon(AOI_PATH)

In [ ]:
# the default aggregation template: located/sharded t-digest, split into
# signal / non-signal photon strata, hive layout at shard order 9, strict-AOI mask on
config = default_config("atl03_tdigest_strata_healpix")
config.output = default_config("atl03_tdigest_healpix_hive").output
config.output["aoi_mask"] = True
# uniform demo-wide centroid budget (matches every 02_write store; the packaged
# template default is 8192)
for _m in config.aggregation["variables"].values():
    if _m.get("kind") == "ragged":
        _m.setdefault("params", {})["delta"] = 4096
config.worker = {"memory": 4096, "extra_disk": True}
config.aggregation["streaming"] = {"mode": "spill"}

## 1 — query: NASA CMR → shard map (ICESat-2 ATL03 v007)

In [7]:
with stage("query"):
    catalog = CMRSource().fetch(Query("ATL03", "007", *DATES, region=AOI_PATH))
    shardmap = ShardMap.build(catalog, from_config(config), region=PARTS, mortie_order=9)
shardmap.to_json(str(OUT / "tldr_shardmap.json"))
print(f"{len(catalog):,} granules -> {shardmap.metadata['total_shards']} shards "
      f"({shardmap.metadata['granules_assigned']:,} assigned, "
      f"{shardmap.metadata['total_pairs']:,} shard-granule pairs)")

INFO:stac_geoparquet.arrow._api:parse_stac_items_to_arrow start | CPU%: 0.0 | CPU_USER_TIME: 1.147 | RSS(MB):202.39 | USS(MB):149.55
INFO:stac_geoparquet.arrow._batch:Items Length: 81
[query] 4.5s
81 granules -> 4 shards (81 assigned, 237 shard-granule pairs)


## 2 — build: one Lambda per shard writes the store

Submit is the shard map plus the template; `wait()` joins the fleet and the post-run tail. The final cell wipes the store when you're done — the hive template guard refuses to re-template a populated store, so skip it only if you're keeping the run.

In [ ]:
with stage("build"):
    run = Run.from_config(config, shardmap=str(OUT / "tldr_shardmap.json"),
                          store=STORE, overwrite=True)
    handle = run.dispatch()
    handle.wait()
print(handle.status() | {"cost_usd": round(handle.cost_usd(), 4)})

## 3 — read: masked shards → 128×128×64 signal tensors → `.npz`

Each populated o12 block (128×128 order-19 cells, 64 elevation bins) lands as one compressed `.npz`: `tensor` (uint32 photon counts), `mask` (0 unobserved / 1 observed-empty / 2 has data), `offset`/`gain` (metres: bin *i* spans `offset + i·gain`), `morton` (the georeference).

In [ ]:
FIELD = f"{config.output['grid']['child_order']}/h_tdigest_signal"
files, photons = [], 0
with stage("read"):
    manifest = read_manifest(STORE)
    for key in shardmap.shard_keys:
        leaf = open_leaf(STORE, int(key), manifest=manifest)
        for tensor, mask, (offset, gain), word in read_tensors(
            leaf, FIELD, n_bins=64, resolution=1.0, block_order=12, fit="degrade_resolution"
        ):
            path = OUT / f"hhdc_{morton_decimal(word)}.npz"
            np.savez_compressed(path, tensor=tensor, mask=mask, offset=np.float64(offset),
                                gain=np.float64(gain), morton=np.uint64(word))
            files.append(path)
            photons += int(tensor.sum())
n_tensors = len(files)
print(f"{n_tensors} tensors (128x128x64), {photons / 1e6:.1f}M signal photons -> {OUT}/hhdc_*.npz")
print(f"{timings['read'] / max(n_tensors, 1):.3f}s per tensor (fetch + decode + rasterize + write)")

In [ ]:
for name, secs in timings.items():
    print(f"{name:>6}  {secs:8.1f}s")
print(f"{'total':>6}  {sum(timings.values()):8.1f}s")
print(f"{'cost':>6}  ${handle.cost_usd():8.4f}  (metered Lambda, billed-duration rollup)")

In [ ]:
# cleanup: wipe the store (and its .status/ sibling -- same string prefix) so the
# notebook reruns cleanly; the hive template guard refuses a populated store
import boto3

def clear_store(prefix):
    bucket, _, key = prefix.removeprefix("s3://").partition("/")
    s3 = boto3.client("s3")
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=key):
        objs = [{"Key": o["Key"]} for o in page.get("Contents", [])]
        for i in range(0, len(objs), 1000):
            s3.delete_objects(Bucket=bucket, Delete={"Objects": objs[i : i + 1000], "Quiet": True})
        n += len(objs)
    return n

print(f"{STORE}: {clear_store(STORE)} objects deleted")